# Data Validation and Cleaning 
---
## Tasks done in this notebook:
- Load dataset 
- Validate schema types and ranges for example validating that amount > 0 or is_fraud is either 0 or 1
- Detect and remove duplicates 
- Verify missing values 
- Cast columns to optimal types
- Generate a validation report 
- Export to data/processed/cleaned_data.csv

---

## Understanding the dataset:

| Column | Data Type | Description | Example Values|
|--------|-----------|-------------|---------------|
| transaction_id | Integer | Unique identifier for each transaction | 1, 2, 3, ...|
| amount | Float | Monetary value of the transaction in currency units | 84.47, 541.82, 237.01 |
| transaction_hour | Integer | Hour of day when transaction occurred (24-hour format) | 0 - 23 |
| merchant_category | Categorical | Type of merchant where purchase was made | Electronics, Travel, Grocery, Food, Clothing |
| foreign_transaction | Binary | Whether transaction occurred in a foreign country | 0 = Domestic, 1 = Foreign |
| location_mismatch | Binary | Whether transaction location differs from cardholder's typical location | 0 = Match, 1 = Mismatch |
| device_trust_score | Integer | Trust score of the device used for transaction (higher = more trusted) | 25 - 99 |
| velocity_last_24h | Integer | Number of transactions by this cardholder in past 24 hours | 0 - 9+ |
| cardholder_age | Integer | Age of the cardholder in years | 0 - 69 |
| is_fraud | Binary | **Target variable:** Whether transaction is fraudulent | 0 = Legitimate, 1 = Fraud |

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings as warn

### Load Dataset

In [3]:
df = pd.read_csv('../data/raw/credit_card_fraud_10k.csv')

### Initial Data Exploration

In [4]:
head = df.head()
print("head:")
print(head)
shape = df.shape
print("shape:", shape)
describe = df.describe()
print("describe:")
print(describe)

head:
   transaction_id  amount  transaction_hour merchant_category  \
0               1   84.47                22       Electronics   
1               2  541.82                 3            Travel   
2               3  237.01                17           Grocery   
3               4  164.33                 4           Grocery   
4               5   30.53                15              Food   

   foreign_transaction  location_mismatch  device_trust_score  \
0                    0                  0                  66   
1                    1                  0                  87   
2                    0                  0                  49   
3                    0                  1                  72   
4                    0                  0                  79   

   velocity_last_24h  cardholder_age  is_fraud  
0                  3              40         0  
1                  1              64         0  
2                  1              61         0  
3               

### Data Validation

In [5]:
missing_values = df.isnull().sum()
if missing_values.any():
    print("Missing values found:")
    print(missing_values[missing_values > 0])
else:
    print("No missing values found.")

data_types = df.dtypes
print("Data types:")
print(data_types)

duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Duplicate rows found: {duplicate_rows}")    
else:
    print("No duplicate rows found.")

numerical_cols = ['amount', 'transaction_hour', 'device_trust_score', 
                  'velocity_last_24h', 'cardholder_age']
for col in numerical_cols:
    print(f"{col}: min = {df[col].min():.2f}, max = {df[col].max():.2f}")
    print(f"{col}: mean = {df[col].mean():.2f}, median = {df[col].median():.2f}, std = {df[col].std():.2f}")

binary_cols = ['foreign_transaction', 'location_mismatch', 'is_fraud']
for col in binary_cols:
    print(f"{col} value counts: ", df[col].value_counts())
    print(f"values in {col} column: {df[col].unique()}")

print(f"Fraudulent transactions percentage (Fraud rate): {df['is_fraud'].mean() * 100:.2f}%")

print(f"Merchant category counts: {df['merchant_category'].nunique()} unique categories")

No missing values found.
Data types:
transaction_id           int64
amount                 float64
transaction_hour         int64
merchant_category          str
foreign_transaction      int64
location_mismatch        int64
device_trust_score       int64
velocity_last_24h        int64
cardholder_age           int64
is_fraud                 int64
dtype: object
No duplicate rows found.
amount: min = 0.00, max = 1471.04
amount: mean = 175.95, median = 122.09, std = 175.39
transaction_hour: min = 0.00, max = 23.00
transaction_hour: mean = 11.59, median = 12.00, std = 6.92
device_trust_score: min = 25.00, max = 99.00
device_trust_score: mean = 61.80, median = 62.00, std = 21.49
velocity_last_24h: min = 0.00, max = 9.00
velocity_last_24h: mean = 2.01, median = 2.00, std = 1.43
cardholder_age: min = 18.00, max = 69.00
cardholder_age: mean = 43.47, median = 44.00, std = 14.98
foreign_transaction value counts:  foreign_transaction
0    9022
1     978
Name: count, dtype: int64
values in foreign_t

**All numerical cols except amount are normally distributed**

**Amount has positive skew**


### Data Quality Assessment

In [6]:
print("Statistical summary of numerical features:")
print(df[numerical_cols].describe())

print("Class imbalance analysis for 'is_fraud':")
fraud_counts = df['is_fraud'].value_counts()
print("Fraudulent vs Non-Fraudulent transactions:")
print(f"Non Fraudulent transactions: {fraud_counts[0]} ({fraud_counts[0] / fraud_counts.sum() * 100:.2f}%)")
print(f"Fraudulent transactions: {fraud_counts[1]} ({fraud_counts[1] / fraud_counts.sum() * 100:.2f}%)")
print(f"Imbalance ratio (Non-Fraudulent to Fraudulent): {fraud_counts[0] / fraud_counts[1]:.2f}:1")


print("Outlier detection:")
targeted_cols = ['amount', 'device_trust_score', 'velocity_last_24h']
for col in targeted_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col}: {len(outliers)} outliers detected")
    print(f"Percentage of outliers in {col}: {len(outliers) / len(df) * 100:.2f}%")

print("Feature correlation analysis with 'is_fraud':")
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('transaction_id')
correlations = df[numeric_cols].corr()['is_fraud'].drop('is_fraud').sort_values(ascending=False)
print(correlations)

Statistical summary of numerical features:
             amount  transaction_hour  device_trust_score  velocity_last_24h  \
count  10000.000000      10000.000000        10000.000000       10000.000000   
mean     175.949849         11.593300           61.798900           2.008900   
std      175.392827          6.922708           21.487053           1.432559   
min        0.000000          0.000000           25.000000           0.000000   
25%       50.905000          6.000000           43.000000           1.000000   
50%      122.095000         12.000000           62.000000           2.000000   
75%      242.480000         18.000000           80.000000           3.000000   
max     1471.040000         23.000000           99.000000           9.000000   

       cardholder_age  
count    10000.000000  
mean        43.468700  
std         14.979147  
min         18.000000  
25%         30.000000  
50%         44.000000  
75%         56.000000  
max         69.000000  
Class imbalance anal

### **Correlation Analysis**

_Positive Correlations:_
- foreign_transaction
- location_mismatch
- velocity_last_24h      
- amount                 

_Approximately Zero Correlations:_
- cardholder_age        

_Negative Correlations:_
- device_trust_score    
- transaction_hour      
---
| Feature | Correlation | Interpretation|
|---------|-------------|---------------|
|foreign_transaction | +0.186 | Strongest predictor. Foreign transactions are approximately 18.6% more likely to be fraudulent. Makes sense: cross-border transactions are harder to verify and commonly associated with stolen card data |
| location_mismatch | +0.173 | Second strongest. When transaction location doesn't match cardholder's typical pattern, fraud risk increases significantly. Geographic anomalies are classic fraud signals|
| velocity_last_24h | +0.103 | More transactions in 24 hours makes higher fraud risk. Fraudsters often test cards with rapid, multiple small transactions before larger attacks |
| amount | +0.028 | Very weak positive link. Suggests fraud occurs across all transaction sizes. Amount alone is not a reliable fraud indicator so both small (or test) charges and large purchases can be fraudulent |
| cardholder_age | -0.0006 | Approximately zero correlation. Age does not predict fraud risk in this dataset as fraud affects all age groups equally |
| device_trust_score | -0.138 | Lower trust scores correlate with higher fraud. Unrecognized or suspicious devices (low scores) are red flags. Inverse relationship: as trust score decreases, fraud risk increases |
| transaction_hour | -0.139 | Negative correlation means fraud is more common at lower hour values (late night/early morning, for example 0-5 AM). Fraudsters often operate during off-hours when cardholders are asleep and monitoring may be reduced |


### Data Cleaning

In [7]:
df_cleaned = df.copy()

if df_cleaned.isnull().any().any():
    print("Handling missing values:")
    for col in df_cleaned.columns:
        if df_cleaned[col].isnull().any():
            if df_cleaned[col].dtype in ['float64', 'int64']:
                median_value = df_cleaned[col].median()
                df_cleaned[col].fillna(median_value, inplace=True)
                print(f"Filled missing values in {col} with median: {median_value}")
            else:
                mode_value = df_cleaned[col].mode()[0]
                df_cleaned[col].fillna(mode_value, inplace=True)
                print(f"Filled missing values in {col} with mode: {mode_value}")
else:
    print("No missing values to handle.")

duplicate_rows = df_cleaned.duplicated().sum()
if duplicate_rows > 0:
    print(f"Handling {duplicate_rows} exact duplicate rows by keeping the first occurrence.")
    df_cleaned.drop_duplicates(inplace=True)
    print(f"Exact duplicates removed. Remaining duplicates: {df_cleaned.duplicated().sum()}")
else:   
    print("No exact duplicate rows to handle.")

if df_cleaned.duplicated(subset=['amount', 'cardholder_age', 'transaction_hour' , 'merchant_category']).sum() > 0:
    print("Handling partial duplicates based on 'amount', 'cardholder_age', 'transaction_hour' and 'merchant_category'.")
    df_cleaned.drop_duplicates(subset=['amount', 'cardholder_age', 'transaction_hour', 'merchant_category'], inplace=True)
    print(f"Partial duplicates removed. Remaining partial duplicates: {df_cleaned.duplicated(subset=['amount', 'cardholder_age', 'transaction_hour', 'merchant_category']).sum()}")
else:
    print("No partial duplicates based on 'amount', 'cardholder_age', 'transaction_hour' and 'merchant_category' to handle.")

for col in numeric_cols:
    if not pd.api.types.is_numeric_dtype(df_cleaned[col]):
        print(f"Converting {col} to numeric.")
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')


df_cleaned['transaction_id'] = df_cleaned['transaction_id'].astype(int)
df_cleaned['transaction_hour'] = df_cleaned['transaction_hour'].astype(int)
df_cleaned['device_trust_score'] = df_cleaned['device_trust_score'].astype(float)
df_cleaned['velocity_last_24h'] = df_cleaned['velocity_last_24h'].astype(float)
df_cleaned['cardholder_age'] = df_cleaned['cardholder_age'].astype(int)
df_cleaned['merchant_category'] = df_cleaned['merchant_category'].astype('category')
df_cleaned['amount'] = df_cleaned['amount'].astype(float)
df_cleaned['foreign_transaction'] = df_cleaned['foreign_transaction'].astype(int)
df_cleaned['location_mismatch'] = df_cleaned['location_mismatch'].astype(int)
df_cleaned['is_fraud'] = df_cleaned['is_fraud'].astype(int)

df_cleaned.rename(columns={'category': 'merchant_category'}, inplace=True)

print("Data types after cleaning:")
print(df_cleaned.dtypes)


df_cleaned.to_csv('../data/processed/credit_card_fraud_10k_cleaned.csv', index=False)

No missing values to handle.
No exact duplicate rows to handle.
No partial duplicates based on 'amount', 'cardholder_age', 'transaction_hour' and 'merchant_category' to handle.
Data types after cleaning:
transaction_id            int64
amount                  float64
transaction_hour          int64
merchant_category      category
foreign_transaction       int64
location_mismatch         int64
device_trust_score      float64
velocity_last_24h       float64
cardholder_age            int64
is_fraud                  int64
dtype: object
